# Interrogating Economic Vulnerability Across U.S. Counties

## Purpose

Investigate patterns, contrasts, anomalies, and relationships within the
U.S. County Economic Stress & Food Insecurity dataset to identify findings
that are both analytically defensible and meaningful to a general audience.

This notebook extends the Tableau exploration by returning to the underlying
county-level data and asking new questions rather than simply describing
existing visualizations.

## Guiding Principle

**ESI is a signal, not a diagnosis.**

The Economic Stress Index identifies relative patterns of economic stress
across U.S. counties. It does not explain why those conditions exist, nor
does it capture every factor that can shape economic vulnerability.

Local conditions may also be influenced by health care access,
transportation, employment opportunities, housing availability, public
policy, demographic change, natural disasters, infrastructure, and other
factors not represented in the index.

Patterns identified here should therefore be treated as starting points for
investigation rather than conclusions about individual counties or the
people who live in them.

## Research Approach

For each potential finding:

1. Identify what the county-level data show.
2. Test whether the pattern is substantial and consistent.
3. Look for exceptions, contrasts, and unexpected cases.
4. Distinguish observation from interpretation.
5. Identify what the data cannot explain.
6. Use external research where appropriate to investigate possible context.
7. Consider whether the finding contributes meaningfully to the larger story.

The goal is not to find every interesting pattern in the dataset.

The goal is to find the few that matter.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/processed/Economic_Stress_US_Counties_Master_ESI_C.xlsx")

df = pd.read_excel(
    DATA_PATH,
    dtype={
        "State_FIPS": str,
        "County_FIPS": str
    }
)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

df.head()

Rows: 3,144
Columns: 37


,FIPS,State_FIPS,County_FIPS,State,County,Poverty_All_Pct,Poverty_Index,Median_Household_Income,Unemployment_Pct,Unemployment_Index,...,FI_Above_SNAP_Threshold_Pct,Cost_Per_Meal,Annual_Food_Budget_Shortfall,RUCC_2023,Metro_Status,Region,Census_Division,Uninsured_Year,Uninsured_Rate,Uninsured_Persons
0,1001,01,001,AL,Autauga County,12.9,0.175,78840.0,2.4,0.093,...,0.556,3.68,6528000.0,2,Metropolitan,South,East South Central,2024,8.1,4094.0
1,1003,01,003,AL,Baldwin County,10.5,0.129,82519.0,2.5,0.099,...,0.545,4.15,29861000.0,3,Metropolitan,South,East South Central,2024,9.5,19236.0
2,1005,01,005,AL,Barbour County,28.1,0.468,45902.0,3.6,0.159,...,0.581,3.53,3181000.0,6,Nonmetropolitan,South,East South Central,2024,11.5,1864.0
3,1007,01,007,AL,Bibb County,21.6,0.343,60359.0,2.7,0.110,...,0.400,3.43,2914000.0,1,Metropolitan,South,East South Central,2024,11.4,1841.0
4,1009,01,009,AL,Blount County,11.7,0.152,69371.0,2.4,0.093,...,0.410,3.44,7167000.0,1,Metropolitan,South,East South Central,2024,12.7,6174.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3144 entries, 0 to 3143
Data columns (total 37 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   FIPS                               3144 non-null   int64  
 1   State_FIPS                         3144 non-null   object 
 2   County_FIPS                        3144 non-null   object 
 3   State                              3144 non-null   object 
 4   County                             3144 non-null   object 
 5   Poverty_All_Pct                    3143 non-null   float64
 6   Poverty_Index                      3143 non-null   float64
 7   Median_Household_Income            3143 non-null   float64
 8   Unemployment_Pct                   3143 non-null   float64
 9   Unemployment_Index                 3143 non-null   float64
 10  Prime_Age_Did_Not_Work_Pct         3143 non-null   float64
 11  Severe_Burden_Pct                  3143 non-null   float

In [3]:
df.columns.tolist()

['FIPS',
 'State_FIPS',
 'County_FIPS',
 'State',
 'County',
 'Poverty_All_Pct',
 'Poverty_Index',
 'Median_Household_Income',
 'Unemployment_Pct',
 'Unemployment_Index',
 'Prime_Age_Did_Not_Work_Pct',
 'Severe_Burden_Pct',
 'High_Burden_35Plus_Pct',
 'Housing_Index',
 'Renter_Pop',
 'Economic_Stress_Score',
 'ESI_Poverty_Percentile',
 'ESI_NonWork_Percentile',
 'ESI_Housing_Percentile',
 'ESI_C',
 'Food_Insecurity_Year',
 'Food_Insecurity_Rate',
 'Food_Insecure_Persons',
 'Child_Food_Insecurity_Rate',
 'Food_Insecure_Children',
 'SNAP_Threshold',
 'FI_At_or_Below_SNAP_Threshold_Pct',
 'FI_Above_SNAP_Threshold_Pct',
 'Cost_Per_Meal',
 'Annual_Food_Budget_Shortfall',
 'RUCC_2023',
 'Metro_Status',
 'Region',
 'Census_Division',
 'Uninsured_Year',
 'Uninsured_Rate',
 'Uninsured_Persons']

In [4]:
df["FIPS_5"] = df["FIPS"].astype(str).str.zfill(5)

df[["FIPS", "FIPS_5", "State", "County"]].head()

,FIPS,FIPS_5,State,County
0,1001,01001,AL,Autauga County
1,1003,01003,AL,Baldwin County
2,1005,01005,AL,Barbour County
3,1007,01007,AL,Bibb County
4,1009,01009,AL,Blount County


In [5]:
extreme_cols = [
    "FIPS_5",
    "State",
    "County",
    "Census_Division",
    "Metro_Status",
    "ESI_C",
    "Poverty_All_Pct",
    "Prime_Age_Did_Not_Work_Pct",
    "Severe_Burden_Pct",
    "Median_Household_Income",
    "Unemployment_Pct",
    "Uninsured_Rate",
    "Food_Insecurity_Rate",
    "Child_Food_Insecurity_Rate"
]

highest_esi = (
    df[extreme_cols]
    .dropna(subset=["ESI_C"])
    .nlargest(15, "ESI_C")
)

highest_esi

,FIPS_5,State,County,Census_Division,Metro_Status,ESI_C,Poverty_All_Pct,Prime_Age_Did_Not_Work_Pct,Severe_Burden_Pct,Median_Household_Income,Unemployment_Pct,Uninsured_Rate,Food_Insecurity_Rate,Child_Food_Insecurity_Rate
1132,22035,LA,East Carroll Parish,West South Central,Nonmetropolitan,99.809100,44.4,53.470790,47.7092,36713.0,13.0,9.6,0.268,0.419
458,13141,GA,Hancock County,South Atlantic,Nonmetropolitan,99.485629,30.7,61.861485,39.0966,43810.0,5.0,14.8,0.195,0.326
150,05077,AR,Lee County,West South Central,Nonmetropolitan,97.974334,41.3,54.941214,27.5391,38287.0,6.2,10.4,0.280,0.445
1177,22125,LA,West Feliciana Parish,West South Central,Metropolitan,97.083466,23.3,47.253693,30.6024,76572.0,3.0,5.7,0.126,0.208
344,12047,FL,Hamilton County,South Atlantic,Nonmetropolitan,96.653940,25.3,49.811250,27.6712,48859.0,5.8,15.4,0.155,0.237
5,01011,AL,Bullock County,East South Central,Nonmetropolitan,96.616820,36.7,29.544341,31.5079,36826.0,3.4,13.2,0.198,0.326
2760,48471,TX,Walker County,West South Central,Nonmetropolitan,96.457737,23.3,40.628524,29.7044,58095.0,4.3,19.6,0.187,0.277
2537,48025,TX,Bee County,West South Central,Nonmetropolitan,96.166083,24.2,41.902296,27.8312,56414.0,4.8,15.5,0.227,0.279
1454,28103,MS,Noxubee County,East South Central,Nonmetropolitan,96.155478,27.4,30.799773,30.8615,41273.0,4.8,14.8,0.212,0.283
2873,51105,VA,Lee County,South Atlantic,Nonmetropolitan,95.874430,26.0,41.278024,26.8149,43916.0,4.4,9.7,0.240,0.299


In [6]:
lowest_esi = (
    df[extreme_cols]
    .dropna(subset=["ESI_C"])
    .nsmallest(15, "ESI_C")
)

lowest_esi

,FIPS_5,State,County,Census_Division,Metro_Status,ESI_C,Poverty_All_Pct,Prime_Age_Did_Not_Work_Pct,Severe_Burden_Pct,Median_Household_Income,Unemployment_Pct,Uninsured_Rate,Food_Insecurity_Rate,Child_Food_Insecurity_Rate
975,20171,KS,Scott County,West North Central,Nonmetropolitan,2.380952,7.1,5.935796,1.7375,76974.0,3.0,13.2,0.162,0.231
1812,35028,NM,Los Alamos County,Mountain,Nonmetropolitan,2.391558,3.8,7.727621,5.1675,149062.0,1.8,2.9,0.069,0.118
2037,38091,ND,Steele County,West North Central,Nonmetropolitan,3.505144,7.4,7.155635,3.7037,89815.0,3.1,6.8,0.104,0.199
1727,31143,NE,Polk County,West North Central,Nonmetropolitan,4.539188,8.0,7.122008,3.9683,76692.0,2.5,7.9,0.128,0.193
2392,46059,SD,Hand County,West North Central,Nonmetropolitan,5.498992,8.4,5.165289,6.0976,73983.0,1.7,7.0,0.123,0.158
2391,46057,SD,Hamlin County,West North Central,Nonmetropolitan,6.267897,8.2,8.884597,3.9702,80325.0,1.8,10.2,0.093,0.137
72,02063,AK,Chugach Census Area,Pacific,Nonmetropolitan,6.331530,7.9,6.791407,8.6086,97265.0,5.9,13.7,0.145,0.196
2393,46061,SD,Hanson County,West North Central,Nonmetropolitan,6.347439,8.7,8.075601,3.1008,87793.0,2.4,9.1,0.084,0.131
850,19119,IA,Lyon County,West North Central,Nonmetropolitan,6.448192,7.3,8.335376,8.6486,85185.0,2.6,5.7,0.092,0.124
1668,31025,NE,Cass County,West North Central,Metropolitan,6.734542,6.2,10.084806,7.7329,92296.0,3.1,6.2,0.118,0.173


### Initial Observations

- Both the highest- and lowest-ESI extremes are dominated by nonmetropolitan
  counties, suggesting that rurality alone does not distinguish high from low
  economic stress.
- High-ESI counties are concentrated heavily in Southern Census divisions,
  while the lowest-ESI counties in this sample are concentrated in the West
  North Central division.
- Similar ESI values can coexist with substantially different uninsured and
  food insecurity rates.
- Scott County, KS and Los Alamos County, NM have nearly identical very-low
  ESI values but markedly different food insecurity and uninsured rates.
- Some very high-ESI counties also have relatively modest food insecurity,
  reinforcing that ESI and food insecurity are related but not equivalent
  constructs.

**Follow-up questions:** What distinguishes high- and low-stress
nonmetropolitan counties? How much do ESI distributions differ by metro
status and Census division? Which counties show the largest divergence
between ESI and food insecurity?

## Investigation 2 — Economic Stress and Metro Status

The highest- and lowest-ESI county groups were both dominated by
nonmetropolitan counties. This section tests whether economic stress
differs systematically between metropolitan and nonmetropolitan counties
across the full dataset.

In [7]:
metro_summary = (
    df.dropna(subset=["ESI_C"])
      .groupby("Metro_Status")
      .agg(
          Counties=("ESI_C", "count"),
          Median_ESI=("ESI_C", "median"),
          Median_Poverty=("Poverty_All_Pct", "median"),
          Median_NonWork=("Prime_Age_Did_Not_Work_Pct", "median"),
          Median_Housing_Burden=("Severe_Burden_Pct", "median"),
          Median_Income=("Median_Household_Income", "median"),
          Median_Unemployment=("Unemployment_Pct", "median"),
          Median_Uninsured=("Uninsured_Rate", "median"),
          Median_Food_Insecurity=("Food_Insecurity_Rate", "median"),
          Median_Child_FI=("Child_Food_Insecurity_Rate", "median")
      )
      .round(3)
)

metro_summary

,Counties,Median_ESI,Median_Poverty,Median_NonWork,Median_Housing_Burden,Median_Income,Median_Unemployment,Median_Uninsured,Median_Food_Insecurity,Median_Child_FI
Metro_Status,,,,,,,,,,
Metropolitan,1185,48.149,11.5,15.339,20.658,76427.0,3.9,8.8,0.149,0.201
Nonmetropolitan,1957,52.407,14.1,18.507,16.044,62644.0,4.1,10.5,0.169,0.233


### Finding 2 — Metro Status Changes the Profile of Economic Stress

Across the 3,142 counties with valid ESI values, nonmetropolitan counties
have a higher median Economic Stress Index than metropolitan counties
(52.4 vs. 48.1).

The difference is not uniform across economic conditions. Nonmetropolitan
counties have higher median poverty, prime-age nonwork, unemployment,
uninsured, overall food insecurity, and child food insecurity rates, along
with substantially lower median household income.

Severe housing burden moves in the opposite direction: the median rate is
higher among metropolitan counties (20.7%) than nonmetropolitan counties
(16.0%).

This suggests that metro status is associated not simply with the level of
economic vulnerability, but with its composition. Rurality should therefore
not be treated as a diagnosis of economic stress: nonmetropolitan counties
appear at both extremes of the ESI distribution, and different forms of
economic strain have different geographies.

## Investigation 3 — Where ESI Does Not Tell the Whole Story

Economic stress and food insecurity are substantially associated at the
county level, but the relationship is incomplete. This section examines
counties where observed food insecurity is substantially higher or lower
than would be expected from the national ESI–food insecurity relationship.

These differences are not treated as errors in the model. Instead, they
identify places where conditions beyond those represented by the ESI may
warrant further investigation.

In [8]:
model_df = (
    df[
        [
            "FIPS_5",
            "State",
            "County",
            "Census_Division",
            "Metro_Status",
            "ESI_C",
            "Food_Insecurity_Rate",
            "Child_Food_Insecurity_Rate",
            "Poverty_All_Pct",
            "Prime_Age_Did_Not_Work_Pct",
            "Severe_Burden_Pct",
            "Median_Household_Income",
            "Unemployment_Pct",
            "Uninsured_Rate"
        ]
    ]
    .dropna(subset=["ESI_C", "Food_Insecurity_Rate"])
    .copy()
)

slope, intercept = np.polyfit(
    model_df["ESI_C"],
    model_df["Food_Insecurity_Rate"],
    1
)

model_df["Predicted_FI"] = (
    intercept + slope * model_df["ESI_C"]
)

model_df["FI_Residual"] = (
    model_df["Food_Insecurity_Rate"] - model_df["Predicted_FI"]
)

print(f"Paired counties: {len(model_df):,}")
print(f"Slope: {slope:.6f}")
print(f"Intercept: {intercept:.6f}")

model_df[
    ["ESI_C", "Food_Insecurity_Rate", "Predicted_FI", "FI_Residual"]
].describe()

Paired counties: 3,142
Slope: 0.001269
Intercept: 0.100042


,ESI_C,Food_Insecurity_Rate,Predicted_FI,FI_Residual
count,3142.000000,3142.000000,3142.000000,3.142000e+03
mean,50.029305,0.163517,0.163517,2.685454e-17
std,20.317528,0.039540,0.025778,2.998172e-02
min,2.380952,0.058000,0.103063,-1.098870e-01
25%,33.872892,0.135000,0.143019,-2.015413e-02
50%,50.376498,0.161000,0.163958,-1.674429e-03
75%,65.779775,0.188000,0.183501,1.825854e-02
max,99.809100,0.371000,0.226676,1.797322e-01


In [9]:
residual_cols = [
    "FIPS_5",
    "State",
    "County",
    "Census_Division",
    "Metro_Status",
    "ESI_C",
    "Food_Insecurity_Rate",
    "Predicted_FI",
    "FI_Residual",
    "Poverty_All_Pct",
    "Prime_Age_Did_Not_Work_Pct",
    "Severe_Burden_Pct",
    "Median_Household_Income",
    "Unemployment_Pct",
    "Uninsured_Rate"
]

higher_than_predicted = (
    model_df[residual_cols]
    .nlargest(10, "FI_Residual")
    .copy()
)

lower_than_predicted = (
    model_df[residual_cols]
    .nsmallest(10, "FI_Residual")
    .copy()
)

higher_than_predicted

,FIPS_5,State,County,Census_Division,Metro_Status,ESI_C,Food_Insecurity_Rate,Predicted_FI,FI_Residual,Poverty_All_Pct,Prime_Age_Did_Not_Work_Pct,Severe_Burden_Pct,Median_Household_Income,Unemployment_Pct,Uninsured_Rate
2655,48261,TX,Kenedy County,West South Central,Nonmetropolitan,52.985470,0.347,0.167268,0.179732,15.5,28.846154,0.0000,46960.0,8.2,23.2
2588,48127,TX,Dimmit County,West South Central,Nonmetropolitan,79.944851,0.371,0.201473,0.169527,28.5,30.694354,17.7119,49127.0,7.0,16.6
2590,48131,TX,Duval County,West South Central,Nonmetropolitan,64.280411,0.329,0.181598,0.147402,24.8,35.090480,2.1889,45438.0,4.6,17.3
2778,48507,TX,Zavala County,West South Central,Nonmetropolitan,70.606639,0.337,0.189625,0.147375,25.7,23.446083,15.4545,41554.0,6.8,17.1
2777,48505,TX,Zapata County,West South Central,Nonmetropolitan,76.911656,0.334,0.197624,0.136376,30.9,25.121261,17.5214,43517.0,6.9,26.8
2713,48377,TX,Presidio County,West South Central,Nonmetropolitan,55.122494,0.305,0.169979,0.135021,19.8,19.189453,11.2040,49703.0,6.5,29.6
2738,48427,TX,Starr County,West South Central,Nonmetropolitan,86.265776,0.344,0.209492,0.134508,35.4,21.428871,24.5621,37375.0,9.5,27.1
2548,48047,TX,Brooks County,West South Central,Nonmetropolitan,55.488387,0.304,0.170443,0.133557,30.6,16.430486,12.5833,39409.0,4.2,18.4
1089,21189,KY,Owsley County,East South Central,Nonmetropolitan,80.496341,0.321,0.202172,0.118828,34.6,37.216117,17.1548,34802.0,5.7,7.6
2564,48079,TX,Cochran County,West South Central,Metropolitan,60.801782,0.288,0.177185,0.110815,20.8,25.886144,7.5988,52164.0,3.9,24.9


### Finding 3A — High Food Insecurity Residuals Are Geographically Concentrated

Eight of the ten counties with the largest positive residuals in the
national ESI–food insecurity model are in Texas. Several appear to form a
South Texas and border-region cluster.

The pattern is not explained simply by unusually high ESI. For example,
Kenedy County has an ESI near the national midpoint (53.0), for which the
model predicts a food insecurity rate of approximately 16.7%. Its observed
food insecurity estimate is 34.7%, a difference of nearly 18 percentage
points.

Several Texas counties with large positive residuals also have high
uninsured rates. This co-occurrence is descriptive and does not establish
that insurance coverage explains the food insecurity residual.

Some unusually low severe-housing-burden values among these counties should
be investigated for source/sample characteristics before interpreting the
ESI or residuals as case-study evidence.

**Follow-up:** Determine whether the Texas concentration persists beyond the
ten most extreme residuals, examine its geographic distribution, and
investigate relevant external context only after validating the pattern.

## Investigation 3B — Is the Residual Pattern Geographically Concentrated?

The most extreme positive food insecurity residuals appear heavily
concentrated in Texas. This section tests whether that pattern persists
across counties rather than relying only on the ten most extreme cases.

In [10]:
state_residuals = (
    model_df
    .groupby("State")
    .agg(
        Counties=("FI_Residual", "count"),
        Mean_Residual=("FI_Residual", "mean"),
        Median_Residual=("FI_Residual", "median"),
        Mean_Actual_FI=("Food_Insecurity_Rate", "mean"),
        Mean_Predicted_FI=("Predicted_FI", "mean")
    )
    .sort_values("Mean_Residual", ascending=False)
)

state_residuals.head(15)

,Counties,Mean_Residual,Median_Residual,Mean_Actual_FI,Mean_Predicted_FI
State,,,,,
AR,75,0.038285,0.037199,0.219573,0.181288
TX,253,0.028734,0.023559,0.197747,0.169013
OK,77,0.028562,0.029953,0.203818,0.175256
KY,120,0.024514,0.020673,0.203750,0.179236
AK,30,0.023584,0.021905,0.168000,0.144416
WY,23,0.022220,0.021384,0.159043,0.136823
NE,93,0.019836,0.019824,0.151806,0.131971
KS,105,0.019801,0.018214,0.159971,0.140171
MS,82,0.016007,0.013851,0.205402,0.189396


In [11]:
texas_residuals = model_df.loc[
    model_df["State"] == "TX",
    ["County", "ESI_C", "Food_Insecurity_Rate", "Predicted_FI", "FI_Residual"]
].copy()

print(f"Texas counties in model: {len(texas_residuals)}")
print(
    f"Count above prediction: "
    f"{(texas_residuals['FI_Residual'] > 0).sum()}"
)
print(
    f"Count below prediction: "
    f"{(texas_residuals['FI_Residual'] < 0).sum()}"
)
print(
    f"Share above prediction: "
    f"{(texas_residuals['FI_Residual'] > 0).mean():.1%}"
)
print(
    f"Median residual: "
    f"{texas_residuals['FI_Residual'].median():.3f}"
)

Texas counties in model: 253
Count above prediction: 201
Count below prediction: 52
Share above prediction: 79.4%
Median residual: 0.024


### Finding 3B — The Positive Residual Pattern Extends Beyond Extreme Counties

The concentration of large positive food insecurity residuals is not limited
to a small number of Texas outliers.

Across 253 Texas counties in the model, 201 (79.4%) have observed food
insecurity rates above the value predicted by the national ESI relationship.
The median Texas residual is approximately +2.4 percentage points, and the
mean is approximately +2.9 points.

Texas, however, does not have the largest state-level residual. Arkansas
ranks highest, with its 75 counties averaging approximately 3.8 percentage
points more food insecurity than predicted and a similar median residual
(+3.7 points), indicating that the result is not driven only by a few
extreme counties.

Other states with relatively large positive mean residuals include Oklahoma,
Kentucky, Alaska, Wyoming, Nebraska, and Kansas. The geographic diversity of
these states cautions against assuming a single regional explanation.

The state pattern establishes where the national ESI–food insecurity
relationship systematically underpredicts observed food insecurity. It does
not establish why these differences occur.

**Follow-up:** Examine the opposite side of the relationship: states and
counties where food insecurity is substantially lower than ESI would predict.
Compare the two patterns before investigating possible external explanations.

In [12]:
state_residuals.tail(15).sort_values("Mean_Residual")

,Counties,Mean_Residual,Median_Residual,Mean_Actual_FI,Mean_Predicted_FI
State,,,,,
NY,62,-0.044192,-0.046528,0.129565,0.173757
RI,5,-0.038634,-0.034854,0.109400,0.148034
NJ,21,-0.035274,-0.037939,0.123571,0.158845
MA,14,-0.033372,-0.031176,0.120286,0.153658
FL,67,-0.031742,-0.032129,0.150851,0.182592
CA,58,-0.030358,-0.035206,0.147655,0.178013
HI,4,-0.029822,-0.027080,0.133000,0.162822
NH,10,-0.029365,-0.027953,0.115300,0.144665
DC,1,-0.027709,-0.027709,0.139000,0.166709


In [13]:
lower_than_predicted

,FIPS_5,State,County,Census_Division,Metro_Status,ESI_C,Food_Insecurity_Rate,Predicted_FI,FI_Residual,Poverty_All_Pct,Prime_Age_Did_Not_Work_Pct,Severe_Burden_Pct,Median_Household_Income,Unemployment_Pct,Uninsured_Rate
2861,51081,VA,Greensville County,South Atlantic,Nonmetropolitan,94.458585,0.110,0.219887,-0.109887,26.2,51.706651,24.6875,56795.0,6.6,8.2
1177,22125,LA,West Feliciana Parish,West South Central,Metropolitan,97.083466,0.126,0.223217,-0.097217,23.3,47.253693,30.6024,76572.0,3.0,5.7
2909,51183,VA,Sussex County,South Atlantic,Metropolitan,75.792767,0.106,0.196205,-0.090205,21.9,37.786010,16.0808,53197.0,3.7,8.5
2834,51025,VA,Brunswick County,South Atlantic,Nonmetropolitan,76.397285,0.109,0.196972,-0.087972,20.2,33.581125,17.2650,53438.0,4.9,10.5
323,12003,FL,Baker County,South Atlantic,Metropolitan,74.318592,0.108,0.194334,-0.086334,13.0,25.883284,25.2312,78227.0,4.2,11.0
1859,36059,NY,Nassau County,Middle Atlantic,Metropolitan,41.107222,0.071,0.152197,-0.081197,5.9,13.476404,27.1153,141824.0,3.3,4.3
386,12129,FL,Wakulla County,South Atlantic,Metropolitan,55.933821,0.092,0.171009,-0.079009,11.8,23.336939,17.7665,77770.0,3.4,11.7
2891,51145,VA,Powhatan County,South Atlantic,Metropolitan,40.826175,0.075,0.151841,-0.076841,6.7,19.873664,18.3246,109889.0,2.7,6.9
2905,51175,VA,Southampton County,South Atlantic,Nonmetropolitan,70.002121,0.114,0.188858,-0.074858,12.7,26.746507,22.3055,70013.0,3.5,7.6
325,12007,FL,Bradford County,South Atlantic,Nonmetropolitan,92.003394,0.142,0.216772,-0.074772,18.2,34.661320,34.5270,58527.0,4.5,13.1


### Finding 3C — ESI Overprediction Also Has Geographic Structure

The opposite side of the model also shows geographic concentration. New York
has the largest negative mean state residual, with county food insecurity
averaging approximately 4.4 percentage points below the national ESI-based
prediction. New Jersey, Massachusetts, New Hampshire, Vermont, and
Pennsylvania also have negative average residuals.

The pattern is not exclusively Northeastern. California, Florida, Virginia,
North Dakota, Minnesota, Iowa, and other states also show substantial
negative average residuals.

At the county level, several of the largest negative residuals occur in
counties with very high ESI values. Greensville County, VA has an ESI of
94.5 but observed food insecurity of 11.0%, compared with approximately
22.0% predicted by the national relationship. West Feliciana Parish, LA
shows a similar divergence: ESI 97.1, observed food insecurity 12.6%, and
predicted food insecurity approximately 22.3%.

Several extreme negative-residual counties also have unusually high
prime-age nonwork rates. This raises a methodological question about whether
the individual ESI components have different relationships with food
insecurity across counties.

**Follow-up:** Examine which ESI components are most strongly associated with
positive and negative food insecurity residuals before seeking external
explanations for individual counties.

## Investigation 4 — ESI Components and Food Insecurity

The Economic Stress Index was designed to represent multiple dimensions of
county-level economic stress, not to optimize prediction of food insecurity.

The residual analysis nevertheless raises an important question: do the
three ESI components—poverty, prime-age nonwork, and severe housing
burden—have different relationships with food insecurity?

This section examines those relationships individually and then investigates
whether particular component profiles are associated with counties where
food insecurity differs substantially from the ESI-based prediction.

In [14]:
components = [
    "Poverty_All_Pct",
    "Prime_Age_Did_Not_Work_Pct",
    "Severe_Burden_Pct"
]

component_correlations = []

for component in components:
    paired = df[[component, "Food_Insecurity_Rate"]].dropna()

    component_correlations.append({
        "Component": component,
        "Counties": len(paired),
        "Pearson_r": paired[component].corr(
            paired["Food_Insecurity_Rate"],
            method="pearson"
        ),
        "Spearman_rho": paired[component].corr(
            paired["Food_Insecurity_Rate"],
            method="spearman"
        )
    })

component_correlations = pd.DataFrame(component_correlations)

component_correlations.round(3)

,Component,Counties,Pearson_r,Spearman_rho
0,Poverty_All_Pct,3142,0.740,0.781
1,Prime_Age_Did_Not_Work_Pct,3142,0.590,0.667
2,Severe_Burden_Pct,3142,-0.014,-0.039


### Finding 4A — ESI Components Have Very Different Relationships With Food Insecurity

The three components of the Economic Stress Index have substantially
different county-level relationships with food insecurity.

Poverty has the strongest relationship with food insecurity
(Pearson r = 0.740; Spearman rho = 0.781). Prime-age nonwork also has a
substantial relationship (Pearson r = 0.590; Spearman rho = 0.667).

Severe housing burden, by contrast, has essentially no bivariate
county-level relationship with food insecurity in this dataset
(Pearson r = -0.014; Spearman rho = -0.039).

This does not imply that housing burden is unimportant to household economic
well-being or food access. The result is ecological and county-level: counties
with higher severe housing burden rates do not, on average, also have higher
food insecurity rates.

The finding reinforces an important distinction in the project design. ESI
was constructed as a multidimensional measure of relative economic stress,
not as a model optimized to predict food insecurity. Its components therefore
need not have equal relationships with food insecurity.

The result also supports the broader finding that different dimensions of
economic vulnerability have different geographies.

**Follow-up:** Examine how the three ESI components relate to the FI residuals.
This can help determine whether particular component profiles systematically
contribute to ESI–food insecurity divergence.

### Investigation 4B — Which Components Are Associated With ESI–FI Divergence?

If counties with particular economic-stress profiles systematically fall
above or below the ESI-based food insecurity prediction, the residuals may
reveal an important limitation of interpreting the composite score alone.

This analysis examines whether each ESI component is associated with the
direction and magnitude of the food insecurity residual.

In [15]:
residual_component_correlations = []

for component in components:
    paired = model_df[[component, "FI_Residual"]].dropna()

    residual_component_correlations.append({
        "Component": component,
        "Counties": len(paired),
        "Pearson_r": paired[component].corr(
            paired["FI_Residual"],
            method="pearson"
        ),
        "Spearman_rho": paired[component].corr(
            paired["FI_Residual"],
            method="spearman"
        )
    })

residual_component_correlations = pd.DataFrame(
    residual_component_correlations
)

residual_component_correlations.round(3)

,Component,Counties,Pearson_r,Spearman_rho
0,Poverty_All_Pct,3142,0.315,0.281
1,Prime_Age_Did_Not_Work_Pct,3142,0.146,0.157
2,Severe_Burden_Pct,3142,-0.433,-0.482


### Finding 4B — The Composition Beneath the ESI Score Matters

The individual ESI components are associated with the direction of the
food insecurity residual in substantially different ways.

Poverty is positively associated with the residual (Pearson r = 0.315;
Spearman rho = 0.281), indicating that counties with higher poverty tend
to have somewhat more food insecurity than the composite ESI relationship
predicts.

Prime-age nonwork shows a smaller positive relationship with the residual
(Pearson r = 0.146; Spearman rho = 0.157).

Severe housing burden shows the strongest relationship and moves in the
opposite direction (Pearson r = -0.433; Spearman rho = -0.482). Counties
with higher severe housing burden therefore tend to have lower observed
food insecurity than would be predicted from their composite ESI alone.

Taken together with the component-to-food-insecurity correlations, this
suggests that the composition beneath an ESI score matters. Poverty,
prime-age nonwork, and severe housing burden represent different dimensions
of economic stress and do not have equivalent relationships with food
insecurity.

This helps explain why counties with similar ESI values can have
substantially different food insecurity outcomes and reinforces the value
of examining ESI components alongside the composite score.

**Methodological caution:** Because the ESI components are used to construct
ESI, and the residual is derived from a model using ESI, these correlations
are diagnostic rather than independent causal evidence. They describe how
the composite behaves in relation to food insecurity.

**Follow-up:** Compare counties with similar ESI values but very different
component profiles and food insecurity outcomes.

### Index Design Context

Severe housing burden was intentionally included in ESI because housing
pressure represents a dimension of economic stress that is not fully
captured by poverty or labor-force measures.

During index development, the geographic distribution of housing burden
was notably different from the distributions of poverty and prime-age
nonwork. In particular, areas with relatively low poverty and nonwork could
still experience substantial housing pressure.

Its weak national relationship with food insecurity should therefore not
be interpreted as evidence that housing burden is an unsuccessful ESI
component. Rather, the result illustrates why a multidimensional economic
stress index can reveal conditions that would be obscured by an index
constructed primarily from variables that move together.

The ESI is intended to broaden the view of economic stress, not reproduce
the geography of food insecurity.

### Investigation 4C — Where Housing Pressure Tells a Different Story

Severe housing burden captures the most extreme housing-cost pressure,
while High_Burden_35Plus_Pct provides a broader measure of renters
experiencing substantial housing burden.

Examining the broader measure allows us to identify counties where housing
costs may represent substantial economic pressure even when poverty and
prime-age nonwork are comparatively low.

This analysis does not alter the ESI. It investigates the broader housing
context surrounding the housing dimension that the index was designed to
preserve.

In [19]:
housing_relationships = (
    df[
        [
            "High_Burden_35Plus_Pct",
            "Severe_Burden_Pct",
            "Poverty_All_Pct",
            "Prime_Age_Did_Not_Work_Pct",
            "Median_Household_Income",
            "Food_Insecurity_Rate"
        ]
    ]
    .corr(method="pearson")
    ["High_Burden_35Plus_Pct"]
    .drop("High_Burden_35Plus_Pct")
    .sort_values(ascending=False)
)

housing_relationships.round(3)

Severe_Burden_Pct             0.867
Median_Household_Income       0.181
Prime_Age_Did_Not_Work_Pct   -0.006
Poverty_All_Pct              -0.008
Food_Insecurity_Rate         -0.043
Name: High_Burden_35Plus_Pct, dtype: float64

### Finding 4C — Housing Pressure Represents a Distinct Dimension of County Stress

The broader housing-burden measure (35% or more) is strongly associated with
severe housing burden (Pearson r = 0.867), indicating that the two measures
capture closely related dimensions of housing-cost pressure.

However, broader housing burden has essentially no bivariate county-level
relationship with poverty (r = -0.008), prime-age nonwork (r = -0.006), or
food insecurity (r = -0.043).

Its relationship with median household income is small but positive
(r = 0.181).

These results reinforce the methodological rationale for retaining housing
burden as a distinct component of economic stress. Housing pressure has a
different county-level distribution from poverty and labor-force
disconnection and therefore captures economic strain that those measures
alone would not reveal.

The absence of a county-level correlation should not be interpreted as an
absence of household-level consequences. These results describe geographic
co-variation among county measures, not the experience of individual
households.

In [20]:
thresholds = {
    "Poverty_25th": df["Poverty_All_Pct"].quantile(0.25),
    "NonWork_25th": df["Prime_Age_Did_Not_Work_Pct"].quantile(0.25),
    "Housing35_75th": df["High_Burden_35Plus_Pct"].quantile(0.75)
}

thresholds

{'Poverty_25th': np.float64(10.350000000000001),
 'NonWork_25th': np.float64(12.98363534784598),
 'Housing35_75th': np.float64(35.65642)}

In [21]:
housing_pressure_counties = (
    df.loc[
        (df["Poverty_All_Pct"] <= thresholds["Poverty_25th"]) &
        (df["Prime_Age_Did_Not_Work_Pct"] <= thresholds["NonWork_25th"]) &
        (df["High_Burden_35Plus_Pct"] >= thresholds["Housing35_75th"]),
        [
            "FIPS_5",
            "State",
            "County",
            "Census_Division",
            "Metro_Status",
            "ESI_C",
            "Poverty_All_Pct",
            "Prime_Age_Did_Not_Work_Pct",
            "High_Burden_35Plus_Pct",
            "Severe_Burden_Pct",
            "Median_Household_Income",
            "Food_Insecurity_Rate"
        ]
    ]
    .sort_values("High_Burden_35Plus_Pct", ascending=False)
)

print(f"Counties meeting all three conditions: {len(housing_pressure_counties)}")

housing_pressure_counties.head(25)

Counties meeting all three conditions: 111


,FIPS_5,State,County,Census_Division,Metro_Status,ESI_C,Poverty_All_Pct,Prime_Age_Did_Not_Work_Pct,High_Burden_35Plus_Pct,Severe_Burden_Pct,Median_Household_Income,Food_Insecurity_Rate
264,08037,CO,Eagle County,Mountain,Nonmetropolitan,36.822569,7.9,8.573612,48.84235,29.4794,111517.0,0.121
1603,30007,MT,Broadwater County,Mountain,Metropolitan,41.573868,8.3,11.580882,48.73786,30.2913,91182.0,0.121
269,08047,CO,Gilpin County,Mountain,Metropolitan,36.249867,7.3,8.285588,46.81529,32.8025,96815.0,0.107
2020,38057,ND,Mercer County,West North Central,Nonmetropolitan,39.744406,8.1,11.785973,45.71429,26.7143,93154.0,0.087
2421,46117,SD,Stanley County,West North Central,Nonmetropolitan,34.887051,7.4,5.020080,45.24715,31.5589,88080.0,0.115
1223,25009,MA,Essex County,New England,Metropolitan,44.729027,9.3,12.429145,43.83046,27.9971,102044.0,0.134
1219,25001,MA,Barnstable County,New England,Metropolitan,43.610139,8.6,12.528824,43.80586,28.4316,88495.0,0.105
245,08001,CO,Adams County,Mountain,Metropolitan,43.827553,9.8,12.017364,43.78344,26.3062,100686.0,0.149
416,13057,GA,Cherokee County,South Atlantic,Metropolitan,35.618836,6.8,11.842597,43.77763,24.0366,116493.0,0.133
1230,25023,MA,Plymouth County,New England,Metropolitan,37.596776,7.3,11.690809,43.51934,25.8226,111910.0,0.086


In [24]:
housing_pressure_counties = (
    df.loc[
        (df["Poverty_All_Pct"] <= thresholds["Poverty_25th"]) &
        (df["Prime_Age_Did_Not_Work_Pct"] <= thresholds["NonWork_25th"]) &
        (df["High_Burden_35Plus_Pct"] >= thresholds["Housing35_75th"]),
        [
            "FIPS_5",
            "State",
            "County",
            "Region",
            "Census_Division",
            "Metro_Status",
            "ESI_C",
            "Poverty_All_Pct",
            "Prime_Age_Did_Not_Work_Pct",
            "High_Burden_35Plus_Pct",
            "Severe_Burden_Pct",
            "Median_Household_Income",
            "Food_Insecurity_Rate"
        ]
    ]
    .sort_values("High_Burden_35Plus_Pct", ascending=False)
)

In [25]:
housing_pressure_geography = (
    housing_pressure_counties
    .groupby(["Region", "Census_Division"])
    .size()
    .reset_index(name="Counties")
    .sort_values("Counties", ascending=False)
)

housing_pressure_geography

,Region,Census_Division,Counties
5,South,South Atlantic,25
7,West,Mountain,21
1,Midwest,West North Central,16
3,Northeast,New England,15
2,Northeast,Middle Atlantic,13
0,Midwest,East North Central,10
8,West,Pacific,6
6,South,West South Central,3
4,South,East South Central,2


In [26]:
housing_pressure_geography = (
    housing_pressure_counties
    .groupby(["Region", "Census_Division"])
    .size()
    .reset_index(name="Counties")
    .sort_values("Counties", ascending=False)
)

housing_pressure_geography

,Region,Census_Division,Counties
5,South,South Atlantic,25
7,West,Mountain,21
1,Midwest,West North Central,16
3,Northeast,New England,15
2,Northeast,Middle Atlantic,13
0,Midwest,East North Central,10
8,West,Pacific,6
6,South,West South Central,3
4,South,East South Central,2


In [27]:
division_totals = (
    df.groupby(["Region", "Census_Division"])
    .size()
    .reset_index(name="Total_Counties")
)

housing_pressure_division_share = (
    housing_pressure_geography
    .merge(
        division_totals,
        on=["Region", "Census_Division"]
    )
)

housing_pressure_division_share["Pct_of_Division"] = (
    100
    * housing_pressure_division_share["Counties"]
    / housing_pressure_division_share["Total_Counties"]
)

housing_pressure_division_share.sort_values(
    "Pct_of_Division",
    ascending=False
).round(1)

,Region,Census_Division,Counties,Total_Counties,Pct_of_Division
3,Northeast,New England,15,68,22.1
4,Northeast,Middle Atlantic,13,150,8.7
1,West,Mountain,21,281,7.5
0,South,South Atlantic,25,588,4.3
6,West,Pacific,6,168,3.6
2,Midwest,West North Central,16,618,2.6
5,Midwest,East North Central,10,437,2.3
7,South,West South Central,3,470,0.6
8,South,East South Central,2,364,0.5


### Finding 4D — Housing Pressure Reveals Stress That Other Indicators Can Miss

To identify counties where housing pressure tells a substantially different
story from poverty and labor-force measures, counties were classified using
national quartiles:

- poverty rate at or below the 25th percentile (≤ 10.35%)
- prime-age nonwork at or below the 25th percentile (≤ 12.98%)
- renter housing burden of 35% or more at or above the 75th percentile
  (≥ 35.66%)

A total of 111 counties met all three conditions. These counties appear in
every U.S. Census Division, demonstrating that the pattern is not exclusive
to a single region.

However, the pattern is highly disproportionate in New England. Fifteen of
New England's 68 counties (22.1%) met the criteria, compared with 8.7% in
the Middle Atlantic and 7.5% in the Mountain division. At the opposite end,
only 0.5% of East South Central and 0.6% of West South Central counties met
the criteria.

The result helps explain why housing burden was retained as a distinct
component of the Economic Stress Index. In some counties, relatively low
poverty and prime-age nonwork rates can suggest comparatively favorable
economic conditions while substantial housing-cost pressure remains visible
in a separate dimension.

The quartile thresholds are exploratory rather than natural boundaries.
They are used here to identify counties with strongly contrasting indicator
profiles, not to define a new categorical measure of economic vulnerability.

## Investigation 5 — Same Score, Different Stress

The Economic Stress Index summarizes three dimensions of county-level
economic vulnerability into a single measure. A composite score, however,
can conceal important differences in the conditions producing it.

This investigation examines counties with similar ESI values but contrasting
component profiles to determine how much economic conditions can differ
beneath comparable index scores.

The purpose is not to identify a "correct" profile of economic stress, but
to demonstrate why the composite score should be interpreted alongside its
underlying measures.

In [28]:
df["ESI_Band"] = (
    np.floor(df["ESI_C"] / 5) * 5
)

esi_band_summary = (
    df.groupby("ESI_Band")
    .agg(
        Counties=("FIPS_5", "count"),
        ESI_Min=("ESI_C", "min"),
        ESI_Max=("ESI_C", "max"),
        Poverty_Min=("Poverty_All_Pct", "min"),
        Poverty_Max=("Poverty_All_Pct", "max"),
        NonWork_Min=("Prime_Age_Did_Not_Work_Pct", "min"),
        NonWork_Max=("Prime_Age_Did_Not_Work_Pct", "max"),
        Housing_Min=("Severe_Burden_Pct", "min"),
        Housing_Max=("Severe_Burden_Pct", "max")
    )
    .reset_index()
)

esi_band_summary.round(2)

,ESI_Band,Counties,ESI_Min,ESI_Max,Poverty_Min,Poverty_Max,NonWork_Min,NonWork_Max,Housing_Min,Housing_Max
0,0.0,4,2.38,4.54,3.8,8.0,5.94,7.73,1.74,5.17
1,5.0,31,5.50,9.99,3.8,10.0,2.63,11.14,1.43,13.62
2,10.0,86,10.11,15.00,4.2,11.8,2.23,15.50,0.00,15.77
3,15.0,118,15.03,19.94,4.3,13.4,0.99,16.39,0.00,18.54
4,20.0,158,20.04,24.95,4.8,15.5,2.05,18.44,0.00,20.13
5,25.0,197,25.03,29.97,4.2,17.9,1.63,21.96,0.00,22.69
6,30.0,236,30.02,34.99,4.4,20.5,3.03,22.04,0.00,31.56
7,35.0,241,35.02,39.98,4.5,20.2,5.48,50.21,0.00,32.80
8,40.0,217,40.03,44.99,5.8,19.0,6.61,36.04,0.00,30.71
9,45.0,260,45.03,49.99,6.5,25.7,7.96,32.70,0.00,34.04


In [29]:
mid_esi = df.loc[
    df["ESI_C"].between(48, 52),
    [
        "FIPS_5",
        "State",
        "County",
        "Region",
        "Census_Division",
        "Metro_Status",
        "ESI_C",
        "ESI_Poverty_Percentile",
        "ESI_NonWork_Percentile",
        "ESI_Housing_Percentile",
        "Poverty_All_Pct",
        "Prime_Age_Did_Not_Work_Pct",
        "Severe_Burden_Pct",
        "Median_Household_Income",
        "Food_Insecurity_Rate"
    ]
].copy()

print(f"Counties with ESI between 48 and 52: {len(mid_esi)}")

Counties with ESI between 48 and 52: 207


In [30]:
component_cols = [
    "ESI_Poverty_Percentile",
    "ESI_NonWork_Percentile",
    "ESI_Housing_Percentile"
]

component_labels = {
    "ESI_Poverty_Percentile": "Poverty",
    "ESI_NonWork_Percentile": "Nonwork",
    "ESI_Housing_Percentile": "Housing"
}

mid_esi["Highest_Component"] = (
    mid_esi[component_cols]
    .idxmax(axis=1)
    .map(component_labels)
)

mid_esi["Highest_Component"].value_counts()

Highest_Component
Housing    83
Nonwork    81
Poverty    43
Name: count, dtype: int64

### Finding 5A — Similar ESI Scores Can Represent Different Forms of Stress

Among 207 counties with ESI values between 48 and 52, the component with
the highest national percentile varied substantially.:

- Housing burden: 83 counties (40.1%)
- Prime-age nonwork: 81 counties (39.1%)
- Poverty: 43 counties (20.8%)

Thus, counties occupying nearly the same position on the composite Economic
Stress Index do not share a single underlying profile. Housing burden was
the most elevated component in roughly two-fifths of these counties, while
prime-age nonwork was most elevated in a similar share and poverty in about
one-fifth.

This demonstrates an important limitation of interpreting a composite score
alone: similar ESI values can summarize materially different combinations
of economic conditions. The index identifies the overall signal, while its
components provide the context needed to understand the form that stress
takes locally.

"Highest component" refers to the component with the highest national county
percentile within each county. It does not imply causation or establish that
the component is the sole source of economic stress.

In [31]:
mid_esi["Distance_From_50"] = abs(mid_esi["ESI_C"] - 50)

representative_profiles = (
    mid_esi
    .sort_values("Distance_From_50")
    .groupby("Highest_Component", group_keys=False)
    .head(10)
    .sort_values(["Highest_Component", "Distance_From_50"])
)

representative_profiles[
    [
        "State",
        "County",
        "Census_Division",
        "Metro_Status",
        "ESI_C",
        "Highest_Component",
        "ESI_Poverty_Percentile",
        "ESI_NonWork_Percentile",
        "ESI_Housing_Percentile",
        "Poverty_All_Pct",
        "Prime_Age_Did_Not_Work_Pct",
        "Severe_Burden_Pct",
        "Median_Household_Income",
        "Food_Insecurity_Rate"
    ]
].round(3)

,State,County,Census_Division,Metro_Status,ESI_C,Highest_Component,ESI_Poverty_Percentile,ESI_NonWork_Percentile,ESI_Housing_Percentile,Poverty_All_Pct,Prime_Age_Did_Not_Work_Pct,Severe_Burden_Pct,Median_Household_Income,Food_Insecurity_Rate
476,GA,Lee County,South Atlantic,Metropolitan,50.011,Housing,0.200,0.353,0.948,9.7,14.530,27.686,89592.0,0.118
1028,KY,Fayette County,East South Central,Metropolitan,49.979,Housing,0.462,0.194,0.843,12.7,12.040,23.920,71046.0,0.169
2235,OR,Multnomah County,Pacific,Metropolitan,50.074,Housing,0.462,0.154,0.886,12.7,11.258,25.144,90423.0,0.153
1529,MO,Howard County,West North Central,Metropolitan,49.926,Housing,0.325,0.313,0.860,11.2,13.942,24.415,60980.0,0.151
2689,TX,Midland County,West South Central,Metropolitan,50.074,Housing,0.325,0.334,0.843,11.2,14.277,23.924,97719.0,0.178
2744,TX,Tarrant County,West South Central,Metropolitan,50.085,Housing,0.353,0.301,0.848,11.5,13.754,24.064,85208.0,0.165
967,KS,Reno County,West North Central,Nonmetropolitan,50.095,Housing,0.395,0.524,0.584,11.9,17.336,19.195,61984.0,0.168
780,IN,Vanderburgh County,East North Central,Metropolitan,50.143,Housing,0.470,0.263,0.771,12.8,13.218,22.406,63717.0,0.175
646,IL,La Salle County,East North Central,Nonmetropolitan,50.148,Housing,0.470,0.432,0.602,12.8,15.836,19.492,72846.0,0.148
2386,SD,Fall River County,West North Central,Nonmetropolitan,49.825,Housing,0.376,0.124,0.995,11.7,10.573,34.038,71364.0,0.167


In [32]:
mid_esi["Component_Spread"] = (
    mid_esi[component_cols].max(axis=1) -
    mid_esi[component_cols].min(axis=1)
)

mid_esi["Component_Spread"].describe().round(3)

count    207.000
mean       0.469
std        0.216
min        0.027
25%        0.285
50%        0.474
75%        0.649
max        0.962
Name: Component_Spread, dtype: float64

In [33]:
mid_esi[
    [
        "State",
        "County",
        "ESI_C",
        "Highest_Component",
        "ESI_Poverty_Percentile",
        "ESI_NonWork_Percentile",
        "ESI_Housing_Percentile",
        "Component_Spread"
    ]
].sort_values(
    "Component_Spread",
    ascending=False
).head(20).round(3)

,State,County,ESI_C,Highest_Component,ESI_Poverty_Percentile,ESI_NonWork_Percentile,ESI_Housing_Percentile,Component_Spread
2646,TX,Jeff Davis County,50.281,Housing,0.470,0.038,1.000,0.962
2654,TX,Kendall County,49.194,Housing,0.029,0.468,0.979,0.950
418,GA,Clay County,48.886,Poverty,0.964,0.467,0.036,0.929
2386,SD,Fall River County,49.825,Housing,0.376,0.124,0.995,0.870
1751,NV,Douglas County,48.648,Housing,0.095,0.404,0.961,0.865
2628,TX,Haskell County,51.946,Poverty,0.936,0.537,0.086,0.850
1815,NM,Mora County,50.223,Poverty,0.846,0.646,0.015,0.832
1047,KY,Hickman County,50.408,Poverty,0.835,0.668,0.010,0.825
3030,WV,Pleasants County,51.845,Nonwork,0.700,0.840,0.016,0.824
1407,MS,Benton County,50.716,Poverty,0.870,0.605,0.046,0.824


### Finding 5B — Component Differences Remain Large Even at Nearly Identical ESI Levels

Among the 207 counties with ESI values between 48 and 52, the median
difference between a county's highest and lowest component percentile was
47.4 percentile points.

The middle 50% of counties had component spreads ranging from 28.5 to
64.9 percentile points. The mean spread was 46.9 points.

This indicates that substantial differences in composition are not limited
to a few unusual counties. Even within a narrow range around the national
middle of the Economic Stress Index, counties commonly reach similar
composite scores through very different combinations of poverty, prime-age
nonwork, and severe housing burden.

The result reinforces the need to interpret the ESI alongside its component
measures. Similar composite scores indicate similar overall positions on
the index, not similar underlying economic conditions.

## Investigation 6 — What Characterizes Counties Where Food Insecurity Deviates from ESI?

Economic Stress Index values explain a substantial portion of county-level
variation in food insecurity, but considerable variation remains.

Previous analysis identified geographic clustering among counties where food
insecurity is substantially higher or lower than predicted by ESI.

This investigation examines whether those residual patterns are also
associated with broader county characteristics not included in the ESI,
including median household income, unemployment, uninsured rates, metro
status, and child food insecurity.

These comparisons are descriptive. Differences among residual groups do not
establish that any characteristic causes food insecurity to be higher or
lower than predicted.

In [34]:
resid_low = model_df["FI_Residual"].quantile(0.20)
resid_high = model_df["FI_Residual"].quantile(0.80)

print(f"20th percentile residual: {resid_low:.4f}")
print(f"80th percentile residual: {resid_high:.4f}")

20th percentile residual: -0.0239
80th percentile residual: 0.0234


In [35]:
model_df["Residual_Group"] = np.select(
    [
        model_df["FI_Residual"] <= resid_low,
        model_df["FI_Residual"] >= resid_high
    ],
    [
        "Lower than predicted",
        "Higher than predicted"
    ],
    default="Middle"
)

model_df["Residual_Group"].value_counts()

Residual_Group
Middle                   1884
Lower than predicted      629
Higher than predicted     629
Name: count, dtype: int64

In [36]:
model_df.columns.tolist()

['FIPS_5',
 'State',
 'County',
 'Census_Division',
 'Metro_Status',
 'ESI_C',
 'Food_Insecurity_Rate',
 'Child_Food_Insecurity_Rate',
 'Poverty_All_Pct',
 'Prime_Age_Did_Not_Work_Pct',
 'Severe_Burden_Pct',
 'Median_Household_Income',
 'Unemployment_Pct',
 'Uninsured_Rate',
 'Predicted_FI',
 'FI_Residual',
 'Residual_Group']

In [37]:
residual_group_summary = (
    model_df
    .groupby("Residual_Group")
    .agg(
        Counties=("FIPS_5", "count"),
        Median_ESI=("ESI_C", "median"),
        Median_FI=("Food_Insecurity_Rate", "median"),
        Median_Child_FI=("Child_Food_Insecurity_Rate", "median"),
        Median_Income=("Median_Household_Income", "median"),
        Median_Poverty=("Poverty_All_Pct", "median"),
        Median_NonWork=("Prime_Age_Did_Not_Work_Pct", "median"),
        Median_Severe_Housing=("Severe_Burden_Pct", "median"),
        Median_Unemployment=("Unemployment_Pct", "median"),
        Median_Uninsured=("Uninsured_Rate", "median"),
        Median_Residual=("FI_Residual", "median")
    )
    .reindex([
        "Lower than predicted",
        "Middle",
        "Higher than predicted"
    ])
)

residual_group_summary.round(3)

,Counties,Median_ESI,Median_FI,Median_Child_FI,Median_Income,Median_Poverty,Median_NonWork,Median_Severe_Housing,Median_Unemployment,Median_Uninsured,Median_Residual
Residual_Group,,,,,,,,,,,
Lower than predicted,629,51.548,0.126,0.178,74565.0,11.7,15.727,22.441,4.0,7.3,-0.036
Middle,1884,48.592,0.160,0.218,67481.0,12.8,16.555,17.747,3.9,9.7,-0.002
Higher than predicted,629,55.165,0.209,0.276,58692.0,15.9,19.549,13.822,4.2,12.5,0.037


### Finding 6A — Counties Above and Below the ESI–Food Insecurity Relationship Have Distinct Profiles

Counties in the highest and lowest quintiles of food-insecurity residuals
showed substantial differences in economic composition and broader context.

The 629 counties with food insecurity higher than predicted by ESI had
median food insecurity of 20.9%, compared with 12.6% among the 629 counties
with food insecurity lower than predicted. Median child food insecurity was
also substantially higher (27.6% versus 17.8%).

The higher-than-predicted group had higher median poverty (15.9% versus
11.7%) and prime-age nonwork (19.5% versus 15.7%). In contrast, severe
housing burden was substantially higher among counties with food insecurity
lower than predicted (22.4% versus 13.8%).

Broader contextual measures also differed. Median household income was
$58,692 in the higher-than-predicted group compared with $74,565 in the
lower-than-predicted group, while the median uninsured rate was 12.5%
compared with 7.3%. Median unemployment rates were similar (4.2% versus
4.0%).

These comparisons reinforce the importance of economic-stress composition.
Counties above and below the national ESI–food insecurity relationship have systematically different economic profiles, reinforcing the importance of examining the conditions underlying the composite score.

The residual groups represent the highest and lowest quintiles of deviations
from a simple ESI–food insecurity regression. These descriptive comparisons
do not establish that any individual characteristic causes counties to fall
above or below the predicted relationship.

In [38]:
model_df[
    [
        "FI_Residual",
        "Uninsured_Rate",
        "Median_Household_Income",
        "Unemployment_Pct"
    ]
].corr(method="pearson")["FI_Residual"].round(3)

FI_Residual                1.000
Uninsured_Rate             0.438
Median_Household_Income   -0.407
Unemployment_Pct           0.146
Name: FI_Residual, dtype: float64

In [39]:
model_df[
    [
        "FI_Residual",
        "Uninsured_Rate",
        "Median_Household_Income",
        "Unemployment_Pct"
    ]
].corr(method="spearman")["FI_Residual"].round(3)

FI_Residual                1.000
Uninsured_Rate             0.435
Median_Household_Income   -0.391
Unemployment_Pct           0.051
Name: FI_Residual, dtype: float64

### Finding 6B — Uninsured Rates and Income Are Associated With Food Insecurity Beyond the ESI Relationship

Among broader contextual measures not included in the Economic Stress Index,
the uninsured rate showed a moderate positive association with food-insecurity
residuals (Pearson r = 0.438; Spearman ρ = 0.435).

Median household income showed a moderate negative association with residuals
(Pearson r = -0.407; Spearman ρ = -0.391). Counties with lower household
incomes therefore tended to have food insecurity farther above the level
predicted by ESI alone.

Unemployment showed a much weaker relationship with residuals
(Pearson r = 0.146; Spearman ρ = 0.051).

These results are consistent with the residual-group comparison, in which
counties with food insecurity higher than predicted had substantially higher
median uninsured rates and lower median household incomes.

The associations do not establish that insurance coverage or income causes
the remaining variation in food insecurity. These measures are related to
other economic, demographic, geographic, and policy conditions that may also
differ across counties.

### Investigation 6C — Broader Context Beyond ESI

The residual analysis identified uninsured rates and median household income
as characteristics associated with counties falling above or below the
food-insecurity level predicted by ESI.

A multivariable diagnostic model is used here to examine whether these
relationships remain visible when ESI, income, uninsured rates, and
unemployment are considered simultaneously.

This model is exploratory and explanatory rather than predictive. Coefficients
describe conditional associations within this county-level dataset and should
not be interpreted as causal effects.

#### Methodological Note — Feeding America Model Inputs

A review of Feeding America's Map the Meal Gap methodology changes the
interpretation of this diagnostic model.

County food-insecurity rates are modeled estimates rather than direct county
measurements. Feeding America's estimation methodology includes poverty,
unemployment, median income, homeownership, disability prevalence,
demographic characteristics, and state and year effects.

Because poverty, median household income, and unemployment contribute to the
construction of the food-insecurity estimates, relationships involving these
variables should not be interpreted as wholly independent validation or as
evidence that they newly explain variation in the modeled outcome.

The increase in R² from the multivariable diagnostic model is therefore
descriptive and partly reflects overlap with variables used in the Map the
Meal Gap estimation process.

Uninsured rate is not listed among the inputs to the overall food-insecurity
estimation model reviewed here. Its association with food-insecurity
residuals therefore remains a useful contextual finding, although it may be
correlated with other modeled or unmeasured county and state characteristics
and should not be interpreted causally.

This methodological audit reinforces the need to distinguish between
relationships discovered across independently measured indicators and
relationships involving variables used to construct a modeled outcome.

In [40]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

diagnostic_cols = [
    "ESI_C",
    "Median_Household_Income",
    "Uninsured_Rate",
    "Unemployment_Pct"
]

diag = model_df[
    diagnostic_cols + ["Food_Insecurity_Rate"]
].dropna().copy()

scaler = StandardScaler()

X_std = scaler.fit_transform(diag[diagnostic_cols])
y_std = StandardScaler().fit_transform(
    diag[["Food_Insecurity_Rate"]]
).ravel()

diagnostic_model = LinearRegression()
diagnostic_model.fit(X_std, y_std)

standardized_coefficients = pd.Series(
    diagnostic_model.coef_,
    index=diagnostic_cols
).sort_values(key=abs, ascending=False)

print(f"N = {len(diag):,}")
print(f"R² = {diagnostic_model.score(X_std, y_std):.3f}")
standardized_coefficients.round(3)

N = 3,142
R² = 0.665


Median_Household_Income   -0.380
Uninsured_Rate             0.313
ESI_C                      0.271
Unemployment_Pct           0.172
dtype: float64

### Finding 6C — Broader Economic Context Adds Substantial Information Beyond ESI

A standardized multivariable diagnostic model combining ESI, median household
income, uninsured rate, and unemployment explained 66.5% of county-level
variation in food insecurity (N = 3,142), compared with approximately 42.5%
for ESI alone.

Standardized coefficients in the four-variable model were:

- Median household income: -0.380
- Uninsured rate: 0.313
- ESI: 0.271
- Unemployment rate: 0.172

Within this model, lower median household income and higher uninsured rates
showed substantial conditional associations with higher food insecurity.
ESI retained a positive association after the broader contextual measures
were included, while unemployment also contributed additional information.

The increase in explained variation indicates that ESI captures an important
but incomplete dimension of the county economic conditions associated with
food insecurity. Measures intentionally retained outside the index provide
substantial additional information.

These standardized coefficients should not be interpreted as causal effects
or as a universal ranking of variable importance. The measures are
interrelated and may also reflect unmeasured geographic, demographic,
institutional, and policy differences among counties.

## Investigation 7 — Geography of the ESI–Food Insecurity Gap

Previous analysis showed that county food-insecurity residuals are
geographically clustered, with substantial differences in average residuals
across states.

This investigation examines whether counties in the highest and lowest
quintiles of food-insecurity residuals are disproportionately concentrated
within particular Census Divisions.

Because Census Divisions contain very different numbers of counties, raw
counts alone can be misleading. Residual-group counts are therefore
normalized by the total number of modeled counties within each division.

The analysis describes geographic concentration. It does not establish why
counties in a particular region are more likely to fall above or below the
ESI–food insecurity relationship.

In [41]:
division_residual_counts = pd.crosstab(
    model_df["Census_Division"],
    model_df["Residual_Group"]
)

division_residual_shares = (
    division_residual_counts
    .div(division_residual_counts.sum(axis=1), axis=0)
    * 100
)

division_residual_summary = pd.concat(
    [
        division_residual_counts.add_suffix("_Count"),
        division_residual_shares.add_suffix("_Pct")
    ],
    axis=1
)

division_residual_summary[
    [
        "Lower than predicted_Count",
        "Middle_Count",
        "Higher than predicted_Count",
        "Lower than predicted_Pct",
        "Middle_Pct",
        "Higher than predicted_Pct"
    ]
].round(1).sort_values(
    "Higher than predicted_Pct",
    ascending=False
)

Residual_Group,Lower than predicted_Count,Middle_Count,Higher than predicted_Count,Lower than predicted_Pct,Middle_Pct,Higher than predicted_Pct
Census_Division,,,,,,
West South Central,19,198,252,4.1,42.2,53.7
East South Central,15,235,114,4.1,64.6,31.3
West North Central,112,382,124,18.1,61.8,20.1
Mountain,53,182,46,18.9,64.8,16.4
Pacific,58,90,19,34.7,53.9,11.4
East North Central,53,353,31,12.1,80.8,7.1
South Atlantic,191,355,42,32.5,60.4,7.1
Middle Atlantic,102,47,1,68.0,31.3,0.7
New England,26,42,0,38.2,61.8,0.0


### Finding 7A — The ESI–Food Insecurity Gap Has a Strong Regional Geography

Counties in the highest and lowest quintiles of food-insecurity residuals
were not evenly distributed across Census Divisions.

The West South Central division showed the strongest concentration of
higher-than-predicted food insecurity. Among modeled counties in the division,
53.7% fell within the national highest quintile of FI residuals, while only
4.1% fell within the lowest quintile.

East South Central also leaned toward higher-than-predicted food insecurity,
with 31.3% of counties in the highest residual quintile and 4.1% in the
lowest.

The opposite pattern appeared in several eastern divisions. In the Middle
Atlantic, 68.0% of counties fell within the lowest residual quintile and only
0.7% within the highest. New England had no counties in the highest residual
quintile, while 38.2% were in the lowest.

The South Atlantic also leaned strongly toward lower-than-predicted food
insecurity (32.5% lower versus 7.1% higher), demonstrating that the residual
pattern cannot be characterized as a simple North–South divide.

West North Central was close to the national 20% expectation on both sides,
while Mountain counties also showed comparatively balanced representation.

These results indicate that the portion of county food insecurity not
captured by the ESI relationship has a pronounced regional geography.
They do not identify the mechanisms responsible for those differences.

In [42]:
wsc = model_df[
    model_df["Census_Division"] == "West South Central"
]

wsc_state_counts = pd.crosstab(
    wsc["State"],
    wsc["Residual_Group"]
)

wsc_state_shares = (
    wsc_state_counts
    .div(wsc_state_counts.sum(axis=1), axis=0)
    * 100
)

wsc_state_summary = pd.concat(
    [
        wsc_state_counts.add_suffix("_Count"),
        wsc_state_shares.add_suffix("_Pct")
    ],
    axis=1
)

wsc_state_summary[
    [
        "Lower than predicted_Count",
        "Middle_Count",
        "Higher than predicted_Count",
        "Lower than predicted_Pct",
        "Middle_Pct",
        "Higher than predicted_Pct"
    ]
].round(1).sort_values(
    "Higher than predicted_Pct",
    ascending=False
)

Residual_Group,Lower than predicted_Count,Middle_Count,Higher than predicted_Count,Lower than predicted_Pct,Middle_Pct,Higher than predicted_Pct
State,,,,,,
AR,0,14,61,0.0,18.7,81.3
OK,0,27,50,0.0,35.1,64.9
TX,12,114,127,4.7,45.1,50.2
LA,7,43,14,10.9,67.2,21.9


### Finding 7B — The West South Central Pattern Is Broad, but Not Uniform

The West South Central concentration of higher-than-predicted food insecurity
was not solely a consequence of Texas's large number of counties.

Arkansas showed the strongest state-level concentration: 61 of 75 modeled
counties (81.3%) fell within the national highest quintile of food-insecurity
residuals, while none fell within the lowest quintile.

Oklahoma showed a similarly widespread pattern, with 50 of 77 counties
(64.9%) in the highest residual quintile and none in the lowest.

In Texas, 127 of 253 counties (50.2%) fell within the highest residual
quintile, compared with 12 counties (4.7%) in the lowest.

Louisiana differed substantially. Only 21.9% of its modeled counties fell
within the highest residual quintile, close to the 20% national reference,
while 10.9% fell within the lowest.

The concentration of positive residuals is therefore particularly widespread
across Arkansas, Oklahoma, and Texas rather than uniformly characteristic of
all West South Central states. This pattern is consistent with earlier
state-level mean residual results and warrants investigation of contextual
factors that may distinguish these states and counties.

### External Evidence — Food Insecurity, Health Coverage, and Financial Strain

Independent research supports a relationship among food insecurity, health
insurance coverage, access to health care, and household financial hardship.

Longitudinal U.S. research using the Medical Expenditure Panel Survey found
that food insecurity was associated with a higher subsequent probability of
being uninsured, reduced access to care, prescription delays, and greater
financial hardship. Other research has found strong associations between
medical debt and food insecurity among U.S. families with children.

These studies do not establish that county uninsured rates cause food
insecurity. Instead, they suggest that food insecurity and health-care
affordability can reflect overlapping pressures on limited household
resources.

In the county analysis, uninsured rate was moderately associated with the
portion of food insecurity not predicted by ESI (Pearson r = 0.438;
Spearman ρ = 0.435). Because uninsured rate is not listed as an input to the
Feeding America food-insecurity estimation model reviewed for this project,
the relationship provides a useful contextual signal for further
investigation.

Interpretation should remain ecological: relationships observed among county
rates cannot be assumed to describe the circumstances of individual
households.

### External Research Leads — West South Central Residual Pattern

Observed pattern:
Arkansas, Oklahoma, and Texas have unusually large shares of counties in the
highest national quintile of food-insecurity residuals. Louisiana does not.

Candidate contextual factors to investigate independently:

- Health insurance coverage and access
- Employment structure and employer-sponsored coverage
- Rurality and geographic access to services
- Household income and wage conditions
- SNAP participation, eligibility, and administrative access
- Medical costs and medical debt
- State safety-net and policy differences
- Feeding America model construction and state effects

These are research leads, not explanations. A factor should enter the article
only if independent evidence supports its relevance to the observed pattern.

### External Evidence — Unemployment Does Not Fully Describe Work

BLS 2024 annual-average data illustrate why unemployment rate alone provides
an incomplete picture of economic participation.

Arkansas had a 2024 unemployment rate of 3.5%, below the national rate of
4.0%. However, calculations from BLS population and labor-force counts show
that approximately 58.2% of Arkansas's civilian noninstitutional population
age 16 and older participated in the labor force, compared with approximately
62.6% nationally.

Oklahoma and Texas demonstrate a different pattern. Oklahoma's calculated
labor-force participation rate was approximately 62.8%, while Texas's was
approximately 64.9%, above the national rate. Both states nevertheless showed
strong concentrations of counties with food insecurity higher than predicted
by ESI.

These comparisons reinforce the distinction between unemployment, labor-force
participation, and economic security. Low unemployment does not necessarily
mean that a large share of the population is working, and high participation
does not establish that available work provides sufficient wages, hours,
stability, or benefits.

The statewide BLS measures use the civilian noninstitutional population age
16 and older and are not directly equivalent to the county-level prime-age
nonwork measure used in ESI. They provide external context for why broader
measures of work were considered in the project.

### External Evidence — Employment Quantity and Employment Quality Are Different

BLS occupational employment and wage data reinforce the distinction between
having employment available and having employment that provides economic
security.

In May 2024, the national mean hourly wage across all occupations was $32.66.
Several labor markets within Arkansas, Oklahoma, Texas, and Louisiana were
substantially below that level, including Little Rock ($26.70), Oklahoma City
($27.95), San Antonio ($28.58), Shreveport ($25.59), and Jonesboro ($24.46).

Lower-paying occupational groups also represented substantial shares of local
employment. For example, food preparation and serving occupations accounted
for 10.2% of employment in Oklahoma City while averaging $14.24 per hour,
compared with $17.32 nationally.

However, wage conditions do not provide a single explanation for the
food-insecurity residual geography. Louisiana also contains labor markets
with relatively low average wages but did not display the same concentration
of higher-than-predicted county food insecurity observed in Arkansas,
Oklahoma, and Texas.

These findings support treating unemployment, labor-force participation,
occupational structure, and wages as distinct dimensions of labor-market
conditions. Employment alone does not establish economic security, while
low wages alone do not explain the observed county food-insecurity patterns.

## Investigation 8 — Metro Status and the ESI–Food Insecurity Gap

Earlier analysis found that nonmetropolitan counties had higher median ESI
and food-insecurity rates than metropolitan counties. This does not establish
whether nonmetropolitan counties also have systematically higher food
insecurity relative to the ESI–FI relationship.

This investigation examines the distribution of food-insecurity residual
groups by metro status.

Because metropolitan and nonmetropolitan counties differ substantially in
number, comparisons are normalized within each metro-status group.

The analysis is descriptive. Metro status may capture or correlate with many
other differences in population structure, employment, access to services,
regional geography, and local economic conditions.

In [43]:
metro_residual_counts = pd.crosstab(

    model_df["Metro_Status"],

    model_df["Residual_Group"]

)

metro_residual_shares = (

    metro_residual_counts

    .div(metro_residual_counts.sum(axis=1), axis=0)

    * 100

)

metro_residual_summary = pd.concat(

    [

        metro_residual_counts.add_suffix("_Count"),

        metro_residual_shares.add_suffix("_Pct")

    ],

    axis=1

)

metro_residual_summary[

    [

        "Lower than predicted_Count",

        "Middle_Count",

        "Higher than predicted_Count",

        "Lower than predicted_Pct",

        "Middle_Pct",

        "Higher than predicted_Pct"

    ]

].round(1)

Residual_Group,Lower than predicted_Count,Middle_Count,Higher than predicted_Count,Lower than predicted_Pct,Middle_Pct,Higher than predicted_Pct
Metro_Status,,,,,,
Metropolitan,347,736,102,29.3,62.1,8.6
Nonmetropolitan,282,1148,527,14.4,58.7,26.9


### Finding 8A — Nonmetropolitan Counties Are Disproportionately Represented Among Higher FI Residuals

Metro status was associated not only with differences in overall economic conditions and food insecurity, but also with the distribution of counties above and below the national ESI–food insecurity relationship.

Among 1,957 nonmetropolitan counties, 26.9% fell within the national highest
quintile of food-insecurity residuals, compared with 8.6% of 1,185
metropolitan counties.

The opposite pattern appeared in the lowest residual quintile. Among
metropolitan counties, 29.3% had food insecurity lower than predicted by ESI,
compared with 14.4% of nonmetropolitan counties.

This extends the earlier finding that nonmetropolitan counties have higher
median ESI and food insecurity. Nonmetropolitan counties are also
disproportionately represented among places where food insecurity exceeds
the level predicted by the national ESI–FI relationship.

However, metro status does not determine residual direction. Most counties
in both groups remained within the middle 60% of residuals, while counties
from both metro categories appeared in both tails.

Metro status should therefore be interpreted as contextual geography rather
than an explanation. It may reflect differences in employment, income,
transportation, service access, population structure, or other local
conditions not captured by ESI.

In [44]:
wsc_states = model_df[
    model_df["State"].isin(["AR", "OK", "TX", "LA"])
].copy()

state_metro_residual = (
    wsc_states
    .groupby(["State", "Metro_Status"])["Residual_Group"]
    .value_counts()
    .unstack(fill_value=0)
)

state_metro_residual["Total"] = state_metro_residual.sum(axis=1)

state_metro_residual["Higher_Pct"] = (
    state_metro_residual["Higher than predicted"]
    / state_metro_residual["Total"]
    * 100
)

state_metro_residual["Lower_Pct"] = (
    state_metro_residual["Lower than predicted"]
    / state_metro_residual["Total"]
    * 100
)

state_metro_residual[
    [
        "Total",
        "Lower than predicted",
        "Higher than predicted",
        "Lower_Pct",
        "Higher_Pct"
    ]
].round(1)

Residual_Group         Total  Lower than predicted  Higher than predicted  \
State Metro_Status                                                          
AR    Metropolitan        17                     0                     12   
      Nonmetropolitan     58                     0                     49   
LA    Metropolitan        37                     7                      2   
      Nonmetropolitan     27                     0                     12   
OK    Metropolitan        18                     0                      7   
      Nonmetropolitan     59                     0                     43   
TX    Metropolitan        86                     5                     26   
      Nonmetropolitan    167                     7                    101   

Residual_Group         Lower_Pct  Higher_Pct  
State Metro_Status                            
AR    Metropolitan           0.0        70.6  
      Nonmetropolitan        0.0        84.5  
LA    Metropolitan          18.9         5.4  
      Nonmetropolitan        0.0        44.4  
OK    Metropolitan           0.0        38.9  
      Nonmetropolitan        0.0        72.9  
TX    Metropolitan           5.8        30.2  
      Nonmetropolitan        4.2        60.5

### Finding 8B — Rurality Amplifies the Residual Pattern but Does Not Explain It

The concentration of higher-than-predicted food insecurity in Arkansas,
Oklahoma, and Texas persisted across both metropolitan and nonmetropolitan
counties.

Nationally, 8.6% of metropolitan counties and 26.9% of nonmetropolitan
counties fell within the highest quintile of FI residuals.

In Arkansas, 70.6% of metropolitan counties and 84.5% of nonmetropolitan
counties were in the higher-residual group. Corresponding shares were 38.9%
and 72.9% in Oklahoma and 30.2% and 60.5% in Texas.

Nonmetropolitan geography therefore substantially amplified the pattern, but
did not account for it. Metropolitan counties in all three states were also
far more likely than metropolitan counties nationally to have FI estimates
above those predicted by the national ESI–FI relationship.

Louisiana provided an important contrast. Only 5.4% of its metropolitan
counties were in the higher-residual group, compared with 44.4% of its
nonmetropolitan counties, suggesting a much stronger metro/nonmetro divide.

Arkansas and Oklahoma also contained no counties in the national lowest
residual quintile, while Texas contained relatively few.

These results describe the geography of model residuals but do not establish
its cause. Feeding America's county FI estimates incorporate state effects
and several socioeconomic inputs, so state-level residual patterns should
not be interpreted as independent evidence of particular state policies or
conditions.

## Investigation 9 — ESI Composition Within Higher-Residual Counties

Investigation 8 found that nonmetropolitan counties were disproportionately
represented among counties with food insecurity higher than predicted by ESI.
However, higher-residual counties occurred in both metropolitan and
nonmetropolitan settings.

This investigation asks whether the composition of ESI differs between those
two geographic groups.

Analysis is restricted to counties in the national highest quintile of
food-insecurity residuals. Median national percentile ranks are compared for
the three ESI components: poverty, prime-age nonwork, and severe housing
burden.

The purpose is descriptive. Differences in component profiles do not identify
causes of food insecurity or explain residual values.

In [46]:
model_df.columns.tolist()

['FIPS_5',
 'State',
 'County',
 'Census_Division',
 'Metro_Status',
 'ESI_C',
 'Food_Insecurity_Rate',
 'Child_Food_Insecurity_Rate',
 'Poverty_All_Pct',
 'Prime_Age_Did_Not_Work_Pct',
 'Severe_Burden_Pct',
 'Median_Household_Income',
 'Unemployment_Pct',
 'Uninsured_Rate',
 'Predicted_FI',
 'FI_Residual',
 'Residual_Group']

In [47]:
model_df = model_df.merge(
    df[
        [
            "FIPS_5",
            "ESI_Poverty_Percentile",
            "ESI_NonWork_Percentile",
            "ESI_Housing_Percentile"
        ]
    ],
    on="FIPS_5",
    how="left",
    validate="one_to_one"
)

In [48]:
model_df[
    [
        "ESI_Poverty_Percentile",
        "ESI_NonWork_Percentile",
        "ESI_Housing_Percentile"
    ]
].isna().sum()

ESI_Poverty_Percentile    0
ESI_NonWork_Percentile    0
ESI_Housing_Percentile    0
dtype: int64

In [50]:
high_resid = model_df[
    model_df["Residual_Group"] == "Higher than predicted"
].copy()

composition_by_metro = (
    high_resid
    .groupby("Metro_Status")
    .agg(
        Counties=("FIPS_5", "count"),
        Median_ESI=("ESI_C", "median"),
        Poverty_Percentile=("ESI_Poverty_Percentile", "median"),
        NonWork_Percentile=("ESI_NonWork_Percentile", "median"),
        Housing_Percentile=("ESI_Housing_Percentile", "median"),
        Median_FI=("Food_Insecurity_Rate", "median"),
        Median_Residual=("FI_Residual", "median")
    )
)

composition_by_metro.round(3)

,Counties,Median_ESI,Poverty_Percentile,NonWork_Percentile,Housing_Percentile,Median_FI,Median_Residual
Metro_Status,,,,,,,
Metropolitan,102,54.542,0.596,0.597,0.382,0.206,0.034
Nonmetropolitan,527,55.165,0.716,0.666,0.245,0.209,0.038


### Finding 9 — Similar Outcomes Can Conceal Different Stress Profiles

Among counties in the national highest quintile of food-insecurity residuals,
metropolitan and nonmetropolitan counties had remarkably similar median
Economic Stress Index scores and food-insecurity rates.

The 102 metropolitan counties had a median ESI of 54.5 and median food
insecurity of 20.6%. The 527 nonmetropolitan counties had a median ESI of
55.2 and median food insecurity of 20.9%. Median residuals were also similar:
+3.4 and +3.8 percentage points, respectively.

However, the composition of economic stress differed.

Higher-residual nonmetropolitan counties had a median poverty percentile of
71.6 and prime-age nonwork percentile of 66.6, compared with 59.6 and 59.7
among metropolitan counties.

Metropolitan counties showed the opposite pattern for severe housing burden.
Their median housing-burden percentile was 38.2, compared with 24.5 among
nonmetropolitan counties.

Thus, counties with similar overall ESI scores and similarly elevated food
insecurity can exhibit different underlying economic-stress profiles.
Nonmetropolitan higher-residual counties were relatively more
poverty- and nonwork-intensive, while metropolitan higher-residual counties
showed relatively greater housing pressure.

These differences are descriptive and do not establish that particular ESI
components cause food insecurity. Rather, they demonstrate why composite
scores should be interpreted alongside their component measures.